### Setup

In [ ]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device]: {DEVICE}")

In [ ]:
DEFAULT_CONFIG = {
    "project": "cnn-redundancy-reduction",
    "architecture_type": "cnn",
    "learning_rate": 1e-3,
    "epochs": 10,
    "batch_size": 64,
    "lambda_strength": 0.0,
    "correlation_mode": "max",
    "correlation_loss": "sum",
    "kernel_size": 3,
    "log_every_steps": 100,
}

print("[defaults]", DEFAULT_CONFIG)

### Dataset

In [ ]:
DATA_DIR = "./data"
NUM_WORKERS = 2

transform = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=transform)
test_ds = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=transform)

print("[train]:", len(train_ds), "[test]:", len(test_ds))
xb, yb = next(iter(DataLoader(train_ds, batch_size=int(DEFAULT_CONFIG["batch_size"]))))
print("[batch]:", tuple(xb.shape), tuple(yb.shape), xb.dtype)

In [ ]:
def get_loaders():
    pass

### Architectures

In [ ]:
class BasicCNN(nn.Module):
    def __init__(self, in_ch=3, num_classes=10, width=64):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, width, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(width)
        self.conv2 = nn.Conv2d(width, width, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(width)
        self.pool1 = nn.MaxPool2d(2)

        self.conv3 = nn.Conv2d(width, width * 2, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn3   = nn.BatchNorm2d(width * 2)
        self.conv4 = nn.Conv2d(width * 2, width * 2, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn4   = nn.BatchNorm2d(width * 2)
        self.pool2 = nn.MaxPool2d(2)

        self.conv5 = nn.Conv2d(width * 2, width * 4, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn5   = nn.BatchNorm2d(width * 4)
        self.pool3 = nn.MaxPool2d(2)

        self.act     = nn.ReLU(inplace=True)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(width * 4, num_classes)

    def forward(self, x):
        convs: list[tuple[str, torch.Tensor]] = []

        c1 = self.conv1(x)
        convs.append(("cnn.conv1", c1))
        x = self.act(self.bn1(c1))

        c2 = self.conv2(x)
        convs.append(("cnn.conv2", c2))
        x = self.act(self.bn2(c2))
        x = self.pool1(x)

        c3 = self.conv3(x)
        convs.append(("cnn.conv3", c3))
        x = self.act(self.bn3(c3))

        c4 = self.conv4(x)
        convs.append(("cnn.conv4", c4))
        x = self.act(self.bn4(c4))
        x = self.pool2(x)

        c5 = self.conv5(x)
        convs.append(("cnn.conv5", c5))
        x = self.act(self.bn5(c5))
        x = self.pool3(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        logits = self.fc(x)
        return logits, convs

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)

        self.skip_conv = None
        self.skip_bn   = None
        if stride != 1 or in_ch != out_ch:
            self.skip_conv = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, padding=0, bias=False)
            self.skip_bn   = nn.BatchNorm2d(out_ch)

        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        convs: list[tuple[str, torch.Tensor]] = []

        identity = x
        if self.skip_conv is not None and self.skip_bn is not None:
            ds = self.skip_conv(x)
            convs.append(("downsample", ds))
            identity = self.skip_bn(ds)

        c1 = self.conv1(x)
        convs.append(("conv1", c1))
        out = self.act(self.bn1(c1))

        c2 = self.conv2(out)
        convs.append(("conv2", c2))
        out = self.bn2(c2)

        out = out + identity
        out = self.act(out)

        return out, convs


class ResNet(nn.Module):
    def __init__(self, in_ch=3, num_classes=10, widths=(64, 128, 256, 512)):
        super().__init__()
        w1, w2, w3, w4 = widths

        self.stem_conv = nn.Conv2d(in_ch, w1, kernel_size=3, stride=1, padding=1, bias=False)
        self.stem_bn   = nn.BatchNorm2d(w1)
        self.stem_act  = nn.ReLU(inplace=True)

        self.layer1 = nn.ModuleList([ResBlock(w1, w1, 1), ResBlock(w1, w1, 1)])
        self.layer2 = nn.ModuleList([ResBlock(w1, w2, 2), ResBlock(w2, w2, 1)])
        self.layer3 = nn.ModuleList([ResBlock(w2, w3, 2), ResBlock(w3, w3, 1)])
        self.layer4 = nn.ModuleList([ResBlock(w3, w4, 2), ResBlock(w4, w4, 1)])

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(w4, num_classes)

    def forward(self, x):
        stem = self.stem_conv(x)
        x = self.stem_act(self.stem_bn(stem))

        convs: list[tuple[str, torch.Tensor]] = []
        convs.append(("resnet.stem", stem))

        for i, block in enumerate(self.layer1):
            x, bconvs = block(x)
            for name, c in bconvs:
                convs.append((f"resnet.layer1.block{i}.{name}", c))
        for i, block in enumerate(self.layer2):
            x, bconvs = block(x)
            for name, c in bconvs:
                convs.append((f"resnet.layer2.block{i}.{name}", c))
        for i, block in enumerate(self.layer3):
            x, bconvs = block(x)
            for name, c in bconvs:
                convs.append((f"resnet.layer3.block{i}.{name}", c))
        for i, block in enumerate(self.layer4):
            x, bconvs = block(x)
            for name, c in bconvs:
                convs.append((f"resnet.layer4.block{i}.{name}", c))

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits = self.fc(x)
        return logits, convs


In [ ]:
class VGG16(nn.Module):
    def __init__(self, in_ch=3, num_classes=10):
        super().__init__()
        cfg = [
            64,
            64,
            "M",
            128,
            128,
            "M",
            256,
            256,
            256,
            "M",
            512,
            512,
            512,
            "M",
            512,
            512,
            512,
            "M",
        ]

        self.layers = nn.ModuleList()
        ch = in_ch
        for v in cfg:
            if v == "M":
                self.layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                out_ch = int(v)
                self.layers.append(nn.Conv2d(ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False))
                self.layers.append(nn.BatchNorm2d(out_ch))
                self.layers.append(nn.ReLU(inplace=True))
                ch = out_ch

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        convs: list[tuple[str, torch.Tensor]] = []

        for i, layer in enumerate(self.layers):
            if isinstance(layer, nn.Conv2d):
                c = layer(x)
                x = c
                convs.append((f"vgg16.layers.{i}", c))
            else:
                x = layer(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits = self.fc(x)
        return logits, convs


In [ ]:
def get_autoencoder_type(autoencoder_type: str):
    match autoencoder_type:
        case "cnn":
            return BasicCNN
        case "resnet":
            return ResNet
        case "vgg16":
            return VGG16
        case _:
            raise ValueError(
                f"Unknown autoencoder_type={autoencoder_type!r}."
            )

### Criterion

In [ ]:
class RedundancyLoss(nn.Module):
    def __init__(
        self,
        lambda_strength: float,
        *,
        kernel_size: int = 3,
        correlation_mode: str = "max",
        correlation_loss: str = "sum",
    ):
        super().__init__()

        kernel_size = int(kernel_size)
        if kernel_size <= 0 or (kernel_size % 2) == 0:
            raise ValueError(f"kernel_size must be an odd positive int, got {kernel_size}")

        self.lambda_strength = float(lambda_strength)
        self.kernel_radius = (kernel_size - 1) // 2
        self.correlation_mode = correlation_mode
        self.correlation_loss = correlation_loss

        self.ce = nn.CrossEntropyLoss()

        self.last_ce_loss = torch.tensor(0.0)
        self.last_corr_total = torch.tensor(0.0)
        self.last_corr_by_layer: dict[str, torch.Tensor] = {}

    def forward(
        self,
        y: torch.Tensor,
        y_pred: torch.Tensor,
        fm_list: list,
    ) -> torch.Tensor:
        ce_loss = self.ce(y_pred, y)

        corr_total = y_pred.new_zeros(())
        corr_by_layer: dict[str, torch.Tensor] = {}

        if fm_list:
            for idx, fm_item in enumerate(fm_list):
                layer_name, feature_map = self._parse_feature_map(fm_item, idx=idx)

                corr_mat = self.luca_fn(
                    feature_map=feature_map,
                    kernel_radius=self.kernel_radius,
                    mode=self.correlation_mode,
                )

                corr_val = self._reduce_corr_matrix(
                    corr_mat,
                    correlation_loss=self.correlation_loss,
                )

                corr_total = corr_total + corr_val
                corr_by_layer[layer_name] = corr_val.detach()

        self.last_ce_loss = ce_loss.detach()
        self.last_corr_total = corr_total.detach()
        self.last_corr_by_layer = corr_by_layer

        return ce_loss + (self.lambda_strength * corr_total)

    @staticmethod
    def _parse_feature_map(fm_item, *, idx: int) -> tuple[str, torch.Tensor]:
        if isinstance(fm_item, torch.Tensor):
            return f"fm_{idx}", fm_item

        if isinstance(fm_item, (tuple, list)):
            if len(fm_item) >= 2 and isinstance(fm_item[0], str) and isinstance(fm_item[1], torch.Tensor):
                return fm_item[0], fm_item[1]
            raise TypeError(
                "fm_list items must be torch.Tensors or tuples like (name: str, feature_map: Tensor)."
            )

        raise TypeError(
            f"Unsupported fm_list item type: {type(fm_item).__name__}. Expected Tensor or tuple/list."
        )

    @staticmethod
    def _reduce_corr_matrix(corr_mat: torch.Tensor, correlation_loss: str) -> torch.Tensor:
        match correlation_loss:
            case "sum":
                return corr_mat.sum()
            case "max":
                return corr_mat.amax()
            case _:
                raise ValueError(
                    f"Unknown correlation_loss={correlation_loss!r}. Expected one of: 'sum', 'max'."
                )

    @staticmethod
    def luca_fn(
        feature_map: torch.Tensor,
        kernel_radius: int,
        mode: str = "max",
    ) -> torch.Tensor:
        if mode not in {"mean", "max"}:
            raise ValueError(f"mode must be 'mean' or 'max', got {mode!r}")

        eps = 1e-6
        B, C, H, W = feature_map.shape
        FM         = feature_map
        R          = kernel_radius
        
        # 1) Z-score each coordinate across batch
        mu  = FM.mean(dim=0, keepdim=True)
        sig = FM.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
        Z   = (FM - mu) / sig # [B, C, H, W]

        # 2) Extract the neighborhood within `kernel_radius` distance of each position
        patch_HW  = (2 * R) + 1
        patch_len = patch_HW * patch_HW
        patches   = F.unfold(Z, kernel_size=patch_HW, padding=R) # [B, C * patch_len, H * W]
        patches   = patches.view(B, C, patch_len, H * W)         # [B, C, patch_len,  H * W]                     

        # 3) The center element in each neighborhood corresponds to (dh=0, dw=0)
        center_k = (patch_len) // 2
        center   = patches[:, :, center_k, :] # [B, C,            H * W]
        patches  = patches[:, :, :,        :] # [B, C, patch_len, H * W]   

        # 4) Build a channel correlation matrix for every c1 (i) and c2 (j)
        corr = torch.einsum('bip,bjkp->ijbp', center, patches) / patch_len # [C1, C2, B, H * W]
        corr = corr * corr                                                 # [C1, C2, B, H * W]

        if mode == "max":
            corr = corr.amax(dim=(2, 3))
        elif mode == "mean":
            corr = corr.mean(dim=(2, 3))
        
        # 5) Remove same-channel pairs
        mask = torch.triu(torch.ones((C, C), dtype=torch.bool, device=DEVICE), diagonal=1)
        corr = corr * mask

        return corr

### Training

In [ ]:
def _load_config(config_or_path: dict | str | None) -> dict:
    if config_or_path is None:
        return {}
    if isinstance(config_or_path, dict):
        return dict(config_or_path)
    if isinstance(config_or_path, str):
        if not config_or_path.endswith(".json"):
            raise ValueError("Only .json config files are supported for now.")
        with open(config_or_path, "r", encoding="utf-8") as f:
            return json.load(f)
    raise TypeError(f"config_or_path must be dict | str | None, got {type(config_or_path).__name__}")

In [ ]:
def _run_name_from_cfg(cfg: dict) -> str:
    return (
        f"{cfg['architecture_type']}"
        f"-ks_{cfg['kernel_size']}"
        f"-lam_{cfg['lambda_strength']}"
        f"-cm_{cfg['correlation_mode']}"
        f"-cl_{cfg['correlation_loss']}"
        f"-lr_{cfg['learning_rate']}"
    )


def train(config_or_path: dict | str | None = None, *, from_sweep: bool = False) -> dict:
    defaults = dict(DEFAULT_CONFIG)

    if from_sweep:
        run = wandb.init(project=defaults["project"], config=defaults)
        merged = {**defaults, **dict(run.config)}
        run.name = _run_name_from_cfg(merged)
    else:
        merged = {**defaults, **_load_config(config_or_path)}
        run = wandb.init(
            project=merged["project"],
            config=merged,
            name=_run_name_from_cfg(merged),
        )

    cfg = run.config

    batch_size = int(cfg["batch_size"])
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    model_cls = get_autoencoder_type(cfg["architecture_type"])
    model = model_cls().to(DEVICE)

    criterion = RedundancyLoss(
        lambda_strength=cfg["lambda_strength"],
        kernel_size=int(cfg["kernel_size"]),
        correlation_mode=cfg["correlation_mode"],
        correlation_loss=cfg["correlation_loss"],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["learning_rate"])

    def _accuracy(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        return (logits.argmax(dim=1) == y).to(torch.float16).mean()

    def _eval() -> tuple[float, float]:
        model.eval()
        loss_sum = 0.0
        acc_sum = 0.0
        n = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                logits, convs = model(xb)
                loss = criterion(y=yb, y_pred=logits, fm_list=convs)
                bs = int(xb.shape[0])
                loss_sum += float(loss.detach().cpu()) * bs
                acc_sum += float(_accuracy(logits, yb).detach().cpu()) * bs
                n += bs
        return loss_sum / max(1, n), acc_sum / max(1, n)

    global_step = 0

    for epoch in range(int(cfg["epochs"])):
        model.train()

        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits, convs = model(xb)
            loss = criterion(y=yb, y_pred=logits, fm_list=convs)
            loss.backward()
            optimizer.step()

            if global_step % int(cfg["log_every_steps"]) == 0:
                log_payload: dict[str, float | int] = {
                    "train/loss": float(loss.detach().cpu()),
                    "train/acc": float(_accuracy(logits, yb).detach().cpu()),
                    "epoch": epoch,
                    "step": global_step,
                    "losses/ce": float(criterion.last_ce_loss.detach().cpu()),
                }

                for layer_name, layer_loss in criterion.last_corr_by_layer.items():
                    log_payload[f"losses/{layer_name}"] = float(layer_loss.detach().cpu())

                wandb.log(log_payload, step=global_step)

            global_step += 1

        test_loss, test_acc = _eval()
        wandb.log(
            {
                "test/loss": test_loss,
                "test/acc": test_acc,
                "epoch": epoch,
                "step": global_step,
            },
            step=global_step,
        )

    metrics = {"test/loss": test_loss, "test/acc": test_acc}
    wandb.finish()
    return metrics


In [ ]:
def _sweep_train() -> None:
    train(from_sweep=True)


SWEEP_PROJECT = DEFAULT_CONFIG["project"]

SWEEP_CONFIG_BASELINE = {
    "method": "grid",
    "metric": {"name": "test/acc", "goal": "maximize"},
    "parameters": {
        "lambda_strength": {"value": 0.0},
        "architecture_type": {"values": ["cnn", "resnet", "vgg16"]},
        "learning_rate": {"values": [1e-3, 5e-2]},
        "kernel_size": {"value": 3},
        "epochs": {"value": 10},
        "batch_size": {"values": [32, 64]},
        "correlation_mode": {"value": "max"},
        "correlation_loss": {"value": "sum"},
        "log_every_steps": {"value": 25},
    },
}

SWEEP_CONFIG_MAIN = {
    "method": "grid",
    "metric": {"name": "test/acc", "goal": "maximize"},
    "parameters": {
        "lambda_strength": {"values": [1e-3, 1e-4]},
        "architecture_type": {"values": ["cnn", "resnet", "vgg16"]},
        "learning_rate": {"values": [1e-3, 5e-2]},
        "kernel_size": {"values": [3, 5]},
        "epochs": {"value": 10},
        "batch_size": {"values": [32, 64]},
        "correlation_mode": {"values": ["mean", "max"]},
        "correlation_loss": {"values": ["sum", "max"]},
        "log_every_steps": {"value": 25},
    },
}

sweep_id = wandb.sweep(SWEEP_CONFIG_MAIN, project=SWEEP_PROJECT)
wandb.agent(sweep_id, function=_sweep_train)